In [ ]:
import torch
from PIL import Image, ImageOps
from transformers import DetrImageProcessor, DetrForObjectDetection
import cv2 
import csv 
from ultralytics import YOLO 

def image_detection(frame):
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    image = ImageOps.exif_transpose(image)
    image = image.convert("RGB")
    
    processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50", revision="no_timm")
    model = DetrForObjectDetection.from_pretrained("isalia99/detr-resnet-50-sku110k")
    model.eval()
    
    inputs = processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    
    target_sizes = torch.tensor([image.size[::-1]])
    results = processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=0.7)[0]
    
    bbox = []
    for box in results["boxes"]:
        box_np = box.cpu().numpy().astype(int)  
        bbox.append(box_np)
    
    return bbox 


with open('data.csv', 'w') as new_file:
    field = ['bbox1', 'bbox2', 'bbox3', 'bbox4', 'object_id']
    csv_writer = csv.DictWriter(new_file, fieldnames=field)
    csv_writer.writeheader()


model = YOLO("yolov8n.pt")
counter = 1 
webcam = cv2.VideoCapture(0) 
dataset_captured = False


while True: 
    ret, frame = webcam.read() 
    
    if not ret:
     break
    
    if not dataset_captured:
     results = model(frame)
     human_detected = False
        
     for box in results[0].boxes: 
            if int(box.cls[0].item()) == 0:
                human_detected = True
                break
        
     if not human_detected:
         bbox = image_detection(frame)
         with open('data.csv', 'a') as file: 
                csv_writer = csv.writer(file)
                for box in bbox: 
                 csv_writer.writerow([box[0], box[1], box[2], box[3], counter]) 
                 counter += 1 
            
         dataset_captured = True

    
    cv2.imshow('Shelf Monitor', frame)
    if dataset_captured:
        break
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

webcam.release()
cv2.destroyAllWindows()

In [ ]:
from ultralytics import YOLO
import cv2
import mediapipe as mp 
import face_recognition

mp_pose_det = mp.solutions.pose.Pose()
def human_face_dataset(frame, hand_bbox):
    height, width = frame.shape[:2]
    x1, y1, x2, y2 = hand_bbox
    width_hand  = x2 - x1
    height_hand = y2-y1 
    approx_par_x1 = max(0, (x1 - width_hand)*3)
    approx_par_y1 = max(0, (y1 - height_hand)*5)
    approx_par_x2 = min(width, (x2 + width_hand)*3)
    approx_par_y2 = min(height,(y2 + height_hand)*5)
    approimate_person = frame[approx_par_y1:approx_par_y2, approx_par_x1:approx_par_x2]
    rgb = cv2.cvtColor(approimate_person, cv2.COLOR_BGR2RGB)
    pose = mp_pose_det.process(rgb)
    
    detection = pose.pose_landmarks.landmark
    nose = detection[0]
    face_height, face_width  = approimate_person.shape[:2]
    landmarks = pose.pose_landmarks.landmark
    left_eye = landmarks[2]   
    right_eye = landmarks[5]  
    nose = landmarks[0]
    temp_width = (left_eye.x - right_eye.x * face_width) -(left_eye.y - right_eye.y * face_height ) 
    face_height = temp_width * 2 
    center_of_face_x = nose.x * face_width
    center_of_face_y = nose.y * face_height
    face_x1 = int(center_of_face_x - face_width/2)
    face_y1 = int(center_of_face_y - face_height/2)
    face_x2 = int(center_of_face_x + face_width/2)
    face_y2 = int(center_of_face_y + face_height/2)
    
    face_bbox = approimate_person[face_y1:face_y2, face_x1:face_x2]
    face_bbox_rgb = cv2.cvtColor(face_bbox, cv2.COLOR_BGR2RGB)
    face_encodings = face_recognition.face_encodings(face_bbox_rgb, face_bbox)
    
    
    with open('Face_data_set.csv', 'a') as f:
        writer = csv.writer(f)
        row = []
        first = 0 
        if  first is 0 : 
            first = 1 
            face_id = 1 
        else :  
            face_id  = face_id + 1 
        row.extend([face_encodings , face_id])
        writer.writerow(row)
    


model = YOLO('yolo-100doh.pt')
webcam = cv2.VideoCapture(0)

while True:
    ret, frame = webcam.read()
    if not ret:
        break
    
    results = model(frame, conf=0.7)
    hands_bbox = []      
    objects_bbox = []    
    Happen_status = False 
    
    for box in results[0].boxes:
        object_id = int(box.cls[0])
        bbox = box.xyxy[0].tolist()
        if object_id == 0:  
            hands_bbox.append(bbox)
        else:
            objects_bbox.append(bbox)
    if  len(hands_bbox) > 0 and len(objects_bbox) > 0 : 
        Happen_status = True 
    if Happen_status is True : 
       human_face_dataset(frame , hands_bbox)
       
    cv2.imshow('Detection', results[0].plot())
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

webcam.release()
cv2.destroyAllWindows()
